<a href="https://colab.research.google.com/github/Cezikarai/Compilador/blob/main/Compilador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Configuração do Ambiente (Setup)
Instalação das dependências necessárias para construir o compilador no Ubuntu/Colab.
Utilizamos o **Flex** (Análise Léxica), **Bison** (Análise Sintática) e a API C++ do **LLVM** (Geração de Código Intermediário).

In [ ]:
!apt-get update
!apt-get install -y flex bison llvm clang nasm llvm-dev

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bison is already the newest version (2:3.8.2+dfsg-1build1).
flex is already the newest version

## 1.1 Verificação de Versões
Validação das ferramentas instaladas para garantir a compatibilidade do ambiente, conforme exigido no roteiro do laboratório.

In [ ]:
!gcc -v
!flex --version
!bison --version
!nasm -v
!clang --version
!llvm-config --version

Using built-in specs.
COLLECT_GCC=gcc
COLLECT_LTO_WRAPPER=/usr/lib/gcc/x86_64-linux-gnu/11/lto-wrapper
OFFLOAD_TARGET_NAMES=nvptx-none:amdgcn-amdhsa
OFFLOAD_TARGET_DEFAULT=1
Target: x86_64-linux-gnu
Configured with: ../src/configure -v --with-pkgversion='Ubuntu 11.4.0-1ubuntu1~22.04.3' --with-bugurl=file:///usr/share/doc/gcc-11/README.Bugs --enable-languages=c,ada,c++,go,brig,d,fortran,objc,obj-c++,m2 --prefix=/usr --with-gcc-major-version-only --program-suffix=-11 --program-prefix=x86_64-linux-gnu- --enable-shared --enable-linker-build-id --libexecdir=/usr/lib --without-included-gettext --enable-threads=posix --libdir=/usr/lib --enable-nls --enable-bootstrap --enable-clocale=gnu --enable-libstdcxx-debug --enable-libstdcxx-time=yes --with-default-libstdcxx-abi=new --enable-gnu-unique-object --disable-vtable-verify --enable-plugin --enable-default-pie --with-system-zlib --enable-libphobos-checking=release --with-target-system-zlib=auto --enable-objc-gc=auto --enable-multiarch --disable-

# 2. Código-Fonte do Compilador Toy
Nesta seção, geramos os arquivos fonte em C++ e as especificações do Flex/Bison diretamente no sistema de arquivos local usando o comando mágico `%%writefile`.

## 2.1 Analisador Léxico (Flex)
O arquivo `lexer.l` utiliza expressões regulares para ler o código `.pas` e dividi-lo em *tokens* (palavras-chave, identificadores, números e operadores). Comentários e espaços em branco são ignorados.

In [ ]:
%%writefile lexer.l
%{
#include "ast.h"
#include "parser.tab.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
%}

%option noyywrap

%%
"program"   { return PROGRAM; }
"begin"     { return BEGIN_TOK; }
"end"       { return END_TOK; }
"var"       { return VAR; }
"integer"   { return INTEGER; }
"write"     { return WRITE; }
"read_arg"  { return READ_ARG; }
"if"        { return IF; }
"then"      { return THEN; }
"else"      { return ELSE; }
"while"     { return WHILE; }
"do"        { return DO; }

[0-9]+      { yylval.num = atoi(yytext); return NUMBER; }
[a-zA-Z_][a-zA-Z0-9_]* { yylval.id = strdup(yytext); return IDENTIFIER; }

":="        { return ASSIGN; }
"+"         { return PLUS; }
"-"         { return MINUS; }
"*"         { return MULT; }
"/"         { return DIV; }
"="         { return EQ; }
"<"         { return LT; }
">"         { return GT; }
";"         { return SEMI; }
":"         { return COLON; }
"."         { return DOT; }
"("         { return LPAREN; }
")"         { return RPAREN; }

[ \t\n\r]+  { /* ignora espaços em branco */ }
.           { printf("Erro Lexico: Caractere nao reconhecido: %s\n", yytext); }
%%

Overwriting lexer.l


## 2.2 Analisador Sintático (Bison)
O arquivo `parser.y` define a Gramática Livre de Contexto (CFG). Ele consome os tokens e verifica a validade estrutural do código. Durante a análise, as ações semânticas constroem a Árvore de Sintaxe Abstrata (AST).

In [ ]:
%%writefile parser.y
%{
#include <iostream>
#include <memory>
#include "ast.h"

void yyerror(const char *s);
int yylex(void);
%}

%union {
    int num;
    char* id;
    class ExprAST* expr;
    class std::vector<class ExprAST*>* stmt_list;
}

%token PROGRAM BEGIN_TOK END_TOK VAR INTEGER WRITE READ_ARG
%token IF THEN ELSE WHILE DO
%token ASSIGN SEMI COLON DOT LPAREN RPAREN
%token PLUS MINUS MULT DIV EQ LT GT

%token <num> NUMBER
%token <id> IDENTIFIER

%type <expr> expression statement
%type <stmt_list> statement_list block

/* Precedencia para resolver o conflito do dangling else */
%nonassoc THEN
%nonassoc ELSE

%left EQ LT GT
%left PLUS MINUS
%left MULT DIV

%%

program:
    PROGRAM IDENTIFIER SEMI var_declaration_part block DOT {
        MainBlock = new BlockAST(*$5);
        delete $5;
        free($2);
        std::cerr << "AST completa construida com sucesso!\n";
    }
    ;

block:
    BEGIN_TOK statement_list END_TOK { $$ = $2; }
    ;

var_declaration_part:
    VAR var_declarations
    | /* vazio */
    ;

var_declarations:
    var_declarations var_declaration
    | var_declaration
    ;

var_declaration:
    IDENTIFIER COLON type SEMI {
        llvm::Function *TheFunction = Builder.GetInsertBlock()->getParent();
        llvm::IRBuilder<> TmpB(&TheFunction->getEntryBlock(), TheFunction->getEntryBlock().begin());
        llvm::AllocaInst* Alloca = TmpB.CreateAlloca(llvm::Type::getInt32Ty(TheContext), nullptr, $1);
        NamedValues[$1] = Alloca;
        free($1);
    }
    ;

type:
    INTEGER
    ;

statement_list:
    statement_list statement SEMI { $$ = $1; if ($2) $$->push_back($2); }
    | statement SEMI { $$ = new std::vector<ExprAST*>(); if ($1) $$->push_back($1); }
    ;

statement:
    IDENTIFIER ASSIGN expression { $$ = new AssignAST($1, $3); free($1); }
    | WRITE LPAREN expression RPAREN { $$ = new PrintExprAST($3); }
    | WHILE expression DO statement { $$ = new WhileAST($2, $4); }
    | IF expression THEN statement ELSE statement { $$ = new IfAST($2, $4, $6); }
    | block { $$ = new BlockAST(*$1); delete $1; }
    | /* vazio */ { $$ = nullptr; }
    ;

expression:
    expression PLUS expression { $$ = new BinaryExprAST('+', $1, $3); }
    | expression MINUS expression { $$ = new BinaryExprAST('-', $1, $3); }
    | expression MULT expression { $$ = new BinaryExprAST('*', $1, $3); }
    | expression DIV expression { $$ = new BinaryExprAST('/', $1, $3); }
    | expression EQ expression { $$ = new BinaryExprAST('=', $1, $3); }
    | expression LT expression { $$ = new BinaryExprAST('<', $1, $3); }
    | expression GT expression { $$ = new BinaryExprAST('>', $1, $3); }
    | LPAREN expression RPAREN { $$ = $2; }
    | NUMBER { $$ = new NumberExprAST($1); }
    | IDENTIFIER { $$ = new VariableExprAST($1); free($1); }
    | READ_ARG { $$ = new ReadArgExprAST(); }
    ;

%%

void yyerror(const char *s) {
    std::cerr << "Erro sintatico: " << s << std::endl;
}

int main(int argc, char **argv) {
    TheModule = std::make_unique<llvm::Module>("MeuMiniPascal", TheContext);

    // main recebe (int argc, char **argv)
    llvm::Type *Int32Ty = llvm::Type::getInt32Ty(TheContext);
    llvm::Type *Int8PtrPtrTy = llvm::PointerType::getUnqual(llvm::PointerType::getUnqual(llvm::Type::getInt8Ty(TheContext)));
    std::vector<llvm::Type*> MainArgs = {Int32Ty, Int8PtrPtrTy};
    llvm::FunctionType *FT = llvm::FunctionType::get(Int32Ty, MainArgs, false);

    llvm::Function *F = llvm::Function::Create(FT, llvm::Function::ExternalLinkage, "main", TheModule.get());
    llvm::BasicBlock *BB = llvm::BasicBlock::Create(TheContext, "entry", F);
    Builder.SetInsertPoint(BB);

    yyparse();

    // Codegen gera todas as instrucoes do bloco principal na memoria
    if (MainBlock) {
        MainBlock->codegen();
    }

    Builder.CreateRet(llvm::ConstantInt::get(TheContext, llvm::APInt(32, 0, true)));
    TheModule->print(llvm::outs(), nullptr);

    return 0;
}

Overwriting parser.y


## 2.3 Automação de Build (Makefile)
Script para orquestrar a compilação: gera o código C a partir do Flex/Bison e compila tudo integrando com as flags nativas da biblioteca do LLVM.

In [ ]:
%%writefile Makefile
CC = g++
# A flag -std=c++17 agora vem no final para sobrescrever configs antigas do LLVM
CFLAGS = `llvm-config --cxxflags` -Wall -std=c++17
LDFLAGS = `llvm-config --ldflags --system-libs --libs core`
FLEX = flex
BISON = bison

all: toy_compiler

parser.tab.c parser.tab.h: parser.y
	$(BISON) -d parser.y

lex.yy.c: lexer.l parser.tab.h
	$(FLEX) lexer.l

toy_compiler: parser.tab.c lex.yy.c
	$(CC) $(CFLAGS) -o toy_compiler parser.tab.c lex.yy.c $(LDFLAGS)

clean:
	rm -f toy_compiler lex.yy.c parser.tab.c parser.tab.h

Overwriting Makefile


## 2.4 Abstract Syntax Tree e Geração LLVM (C++)
O arquivo `ast.h` define as classes dos nós da árvore (ex: `WhileAST`, `BinaryExprAST`). Cada nó possui um método `codegen()` que traduz a lógica para instruções reais de código de máquina (LLVM IR) usando o `IRBuilder`.

In [ ]:
%%writefile ast.h
#pragma once

#include <llvm/IR/LLVMContext.h>
#include <llvm/IR/IRBuilder.h>
#include <llvm/IR/Module.h>
#include <llvm/IR/Verifier.h>
#include <string>
#include <vector>
#include <map>
#include <memory>
#include <iostream>

inline llvm::LLVMContext TheContext;
inline llvm::IRBuilder<> Builder(TheContext);
inline std::unique_ptr<llvm::Module> TheModule;
inline std::map<std::string, llvm::AllocaInst*> NamedValues;

class ExprAST {
public:
    virtual ~ExprAST() = default;
    virtual llvm::Value* codegen() = 0;
};

inline ExprAST* MainBlock = nullptr;

class NumberExprAST : public ExprAST {
    int Val;
public:
    NumberExprAST(int Val) : Val(Val) {}
    llvm::Value* codegen() override {
        return llvm::ConstantInt::get(TheContext, llvm::APInt(32, Val, true));
    }
};

class VariableExprAST : public ExprAST {
    std::string Name;
public:
    VariableExprAST(const std::string &Name) : Name(Name) {}
    llvm::Value* codegen() override {
        llvm::AllocaInst* Alloca = NamedValues[Name];
        if (!Alloca) {
            std::cerr << "Erro: Variavel " << Name << " nao declarada!\n";
            return nullptr;
        }
        return Builder.CreateLoad(Alloca->getAllocatedType(), Alloca, Name.c_str());
    }
};

class AssignAST : public ExprAST {
    std::string Name;
    ExprAST *Val;
public:
    AssignAST(const std::string &Name, ExprAST *Val) : Name(Name), Val(Val) {}
    ~AssignAST() { delete Val; }
    llvm::Value* codegen() override {
        llvm::Value *V = Val->codegen();
        llvm::AllocaInst* Alloca = NamedValues[Name];
        if (!Alloca) return nullptr;
        Builder.CreateStore(V, Alloca);
        return V;
    }
};

class BinaryExprAST : public ExprAST {
    char Op;
    ExprAST *LHS, *RHS;
public:
    BinaryExprAST(char Op, ExprAST *LHS, ExprAST *RHS) : Op(Op), LHS(LHS), RHS(RHS) {}
    ~BinaryExprAST() { delete LHS; delete RHS; }
    llvm::Value* codegen() override {
        llvm::Value *L = LHS->codegen();
        llvm::Value *R = RHS->codegen();
        if (!L || !R) return nullptr;

        switch (Op) {
            case '+': return Builder.CreateAdd(L, R, "addtmp");
            case '-': return Builder.CreateSub(L, R, "subtmp");
            case '*': return Builder.CreateMul(L, R, "multmp");
            case '/': return Builder.CreateSDiv(L, R, "divtmp");
            case '<':
                L = Builder.CreateICmpSLT(L, R, "cmptmp");
                return Builder.CreateZExt(L, llvm::Type::getInt32Ty(TheContext), "booltmp");
            case '>':
                L = Builder.CreateICmpSGT(L, R, "cmptmp");
                return Builder.CreateZExt(L, llvm::Type::getInt32Ty(TheContext), "booltmp");
            case '=':
                L = Builder.CreateICmpEQ(L, R, "cmptmp");
                return Builder.CreateZExt(L, llvm::Type::getInt32Ty(TheContext), "booltmp");
            default: return nullptr;
        }
    }
};

class PrintExprAST : public ExprAST {
    ExprAST* Arg;
public:
    PrintExprAST(ExprAST* Arg) : Arg(Arg) {}
    ~PrintExprAST() { delete Arg; }
    llvm::Value* codegen() override {
        llvm::Value* V = Arg->codegen();
        if (!V) return nullptr;
        llvm::Function* PrintfFn = TheModule->getFunction("printf");
        if (!PrintfFn) {
            llvm::FunctionType* PrintfType = llvm::FunctionType::get(
                llvm::Type::getInt32Ty(TheContext),
                llvm::PointerType::getUnqual(llvm::Type::getInt8Ty(TheContext)), true);
            PrintfFn = llvm::Function::Create(PrintfType, llvm::Function::ExternalLinkage, "printf", TheModule.get());
        }
        llvm::Value* FormatStr = Builder.CreateGlobalStringPtr("%d\n");
        return Builder.CreateCall(PrintfFn, {FormatStr, V}, "printfcall");
    }
};

class ReadArgExprAST : public ExprAST {
public:
    llvm::Value* codegen() override {
        llvm::Function *TheFunction = Builder.GetInsertBlock()->getParent();
        auto arg_it = TheFunction->arg_begin();
        arg_it++;

        llvm::Value *Argv = arg_it;
        llvm::Value *Argv1Ptr = Builder.CreateGEP(
            llvm::PointerType::getUnqual(llvm::Type::getInt8Ty(TheContext)),
            Argv,
            llvm::ConstantInt::get(TheContext, llvm::APInt(32, 1)),
            "argv1ptr"
        );
        llvm::Value *Argv1 = Builder.CreateLoad(
            llvm::PointerType::getUnqual(llvm::Type::getInt8Ty(TheContext)),
            Argv1Ptr,
            "argv1"
        );

        llvm::Function* AtoiFn = TheModule->getFunction("atoi");
        if (!AtoiFn) {
            llvm::FunctionType* AtoiType = llvm::FunctionType::get(
                llvm::Type::getInt32Ty(TheContext),
                llvm::PointerType::getUnqual(llvm::Type::getInt8Ty(TheContext)), false);
            AtoiFn = llvm::Function::Create(AtoiType, llvm::Function::ExternalLinkage, "atoi", TheModule.get());
        }
        return Builder.CreateCall(AtoiFn, {Argv1}, "readarg");
    }
};

class BlockAST : public ExprAST {
    std::vector<ExprAST*> Statements;
public:
    BlockAST(std::vector<ExprAST*> Statements) : Statements(std::move(Statements)) {}
    ~BlockAST() { for (auto* s : Statements) delete s; }
    llvm::Value* codegen() override {
        for (auto *Stmt : Statements) {
            if (Stmt) Stmt->codegen();
        }
        return llvm::Constant::getNullValue(llvm::Type::getInt32Ty(TheContext));
    }
};

class WhileAST : public ExprAST {
    ExprAST *Cond, *Body;
public:
    WhileAST(ExprAST *Cond, ExprAST *Body) : Cond(Cond), Body(Body) {}
    ~WhileAST() { delete Cond; delete Body; }
    llvm::Value* codegen() override {
        llvm::Function *TheFunction = Builder.GetInsertBlock()->getParent();
        llvm::BasicBlock *CondBB = llvm::BasicBlock::Create(TheContext, "cond", TheFunction);
        llvm::BasicBlock *LoopBB = llvm::BasicBlock::Create(TheContext, "loop", TheFunction);
        llvm::BasicBlock *AfterBB = llvm::BasicBlock::Create(TheContext, "after", TheFunction);

        Builder.CreateBr(CondBB);
        Builder.SetInsertPoint(CondBB);
        llvm::Value *CondV = Cond->codegen();
        CondV = Builder.CreateICmpNE(CondV, llvm::ConstantInt::get(TheContext, llvm::APInt(32, 0)), "ifcond");
        Builder.CreateCondBr(CondV, LoopBB, AfterBB);

        Builder.SetInsertPoint(LoopBB);
        if(Body) Body->codegen();
        Builder.CreateBr(CondBB);

        Builder.SetInsertPoint(AfterBB);
        return llvm::Constant::getNullValue(llvm::Type::getInt32Ty(TheContext));
    }
};

class IfAST : public ExprAST {
    ExprAST *Cond, *Then, *Else;
public:
    IfAST(ExprAST *Cond, ExprAST *Then, ExprAST *Else) : Cond(Cond), Then(Then), Else(Else) {}
    ~IfAST() { delete Cond; delete Then; if(Else) delete Else; }
    llvm::Value* codegen() override {
        llvm::Value *CondV = Cond->codegen();
        CondV = Builder.CreateICmpNE(CondV, llvm::ConstantInt::get(TheContext, llvm::APInt(32, 0)), "ifcond");

        llvm::Function *TheFunction = Builder.GetInsertBlock()->getParent();
        llvm::BasicBlock *ThenBB = llvm::BasicBlock::Create(TheContext, "then", TheFunction);
        llvm::BasicBlock *ElseBB = llvm::BasicBlock::Create(TheContext, "else");
        llvm::BasicBlock *MergeBB = llvm::BasicBlock::Create(TheContext, "ifcont");

        Builder.CreateCondBr(CondV, ThenBB, ElseBB);

        Builder.SetInsertPoint(ThenBB);
        if (Then) Then->codegen();
        Builder.CreateBr(MergeBB);

        // CORREÇÃO: Utilizando getBasicBlockList().push_back() em vez de insert()
        TheFunction->getBasicBlockList().push_back(ElseBB);
        Builder.SetInsertPoint(ElseBB);
        if (Else) Else->codegen();
        Builder.CreateBr(MergeBB);

        // CORREÇÃO: Utilizando getBasicBlockList().push_back() em vez de insert()
        TheFunction->getBasicBlockList().push_back(MergeBB);
        Builder.SetInsertPoint(MergeBB);
        return llvm::Constant::getNullValue(llvm::Type::getInt32Ty(TheContext));
    }
};

Overwriting ast.h


# 3. Algoritmos de Teste de Aceitação
Os arquivos abaixo validam as capacidades estruturais e matemáticas do nosso compilador. Eles testam laços de repetição (`while`), condicionais (`if/else`), operadores matemáticos e a leitura de argumentos do terminal.

### Teste 1: Sequência de Fibonacci (`fibonacci.pas`)
Calcula e imprime o n-ésimo número da sequência.

In [ ]:
%%writefile fibonacci.pas
program FibonacciTest;
var
    n: integer;
    a: integer;
    b: integer;
    temp: integer;
    i: integer;
begin
    n := read_arg;
    a := 0;
    b := 1;
    i := 1;

    if n = 0 then
        write(a)
    else
    begin
        while i < n do
        begin
            temp := a + b;
            a := b;
            b := temp;
            i := i + 1;
        end;
        write(b);
    end;
end.

Overwriting fibonacci.pas


### Teste 2: Fatoração de Inteiros (`factor.pas`)
Recebe um número inteiro e imprime todos os seus fatores primos divisíveis.

In [ ]:
%%writefile factor.pas
program FactorTest;
var
    n: integer;
    d: integer;
    q: integer;
    rem: integer;
begin
    n := read_arg;
    d := 2;

    while n > 1 do
    begin
        q := n / d;
        rem := n - (q * d);

        if rem = 0 then
        begin
            write(d);
            n := q;
        end
        else
        begin
            d := d + 1;
        end;
    end;
end.

Overwriting factor.pas


### Teste 3: Validador de Números Primos (`isprime.pas`)
Retorna `1` (True) se o número for primo, ou `0` (False) caso não seja.

In [ ]:
%%writefile isprime.pas
program IsPrimeTest;
var
    n: integer;
    d: integer;
    q: integer;
    rem: integer;
    is_prime: integer;
begin
    n := read_arg;
    is_prime := 1;

    if n < 2 then
        is_prime := 0
    else
    begin
        d := 2;
        while d < n do
        begin
            q := n / d;
            rem := n - (q * d);

            if rem = 0 then
                is_prime := 0
            else
            begin
                q := q;
            end;

            d := d + 1;
        end;
    end;

    write(is_prime);
end.

Overwriting isprime.pas


### Teste 4: Aproximação dos Dígitos de Pi (`pidigits.pas`)
Utiliza a fração de aproximação 355/113 para calcular `n` casas decimais de Pi iterativamente.

In [ ]:
%%writefile pidigits.pas
program PiDigitsTest;
var
    n: integer;
    pi_num: integer;
    pi_den: integer;
    digit: integer;
    i: integer;
    rem: integer;
begin
    n := read_arg;
    pi_num := 355;
    pi_den := 113;
    i := 0;

    while i < n do
    begin
        digit := pi_num / pi_den;
        write(digit);

        rem := pi_num - (digit * pi_den);
        pi_num := rem * 10;

        i := i + 1;
    end;
end.

Overwriting pidigits.pas


# 4. Execução e Validação
Criação do script bash para realizar a limpeza (`make clean`), construir o compilador, converter os programas de teste para LLVM IR (`.ll`), transformá-los em código Assembly nativo (`.s`) e finalmente extrair os executáveis finais para o sistema operacional.

In [ ]:
%%writefile run_tests.sh
#!/bin/bash

echo "🚀 Iniciando o build do Compilador Toy..."
make clean
make

echo -e "\n📦 Compilando programas de teste (Gerando LLVM IR, Assembly e Binarios)..."

for prog in factor isprime pidigits fibonacci; do
    echo "-> Processando $prog.pas"
    # 1. Gera LLVM IR
    ./toy_compiler < $prog.pas > $prog.ll
    # 2. Gera arquivo Assembly (.s) nativo
    clang -S -masm=intel $prog.ll -o $prog.s
    # 3. Gera o executável nativo
    clang $prog.ll -o ${prog}_exe
done

echo -e "\n✅ Tudo compilado com sucesso!"
echo "Para testar, execute no terminal:"
echo " ./factor_exe 84"
echo " ./isprime_exe 17"
echo " ./pidigits_exe 6"
echo " ./fibonacci_exe 10"

Overwriting run_tests.sh


## 4.1 Evidência de Funcionamento
Execução dos binários nativos de Linux gerados pelo nosso compilador, passando argumentos dinâmicos pelo terminal (`argv`).

In [ ]:
# Dá permissão de execução ao script e roda o pipeline de build
!chmod +x run_tests.sh
!./run_tests.sh

!echo ""
!echo "=== RESULTADOS FINAIS DOS ALGORITMOS ==="

# Teste 1: Fatorando 84 (Esperado: 2, 2, 3, 7)
!echo "Fatores de 84:" && ./factor_exe 84

# Teste 2: Checando se 17 é primo (Esperado: 1)
!echo -e "\nO numero 17 é primo? (1=Sim, 0=Nao):" && ./isprime_exe 17

# Teste 3: Primeiros 6 dígitos de Pi (Esperado: 3 1 4 1 5 9)
!echo -e "\nPrimeiros 6 digitos de Pi:" && ./pidigits_exe 6

# Teste 4: 10º número de Fibonacci (Esperado: 55)
!echo -e "\n10º numero da Sequencia de Fibonacci:" && ./fibonacci_exe 10

🚀 Iniciando o build do Compilador Toy...
rm -f toy_compiler lex.yy.c parser.tab.c parser.tab.h
bison -d parser.y
flex lexer.l
g++ `llvm-config --cxxflags` -Wall -std=c++17 -o toy_compiler parser.tab.c lex.yy.c `llvm-config --ldflags --system-libs --libs core`
lex.yy.c:1263:17: warning: ‘void yyunput(int, char*)’ defined but not used []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-function-Wunused-function]8;;]
 1263 |     static void yyunput (int c, char * yy_bp )
      |                 ^~~~~~~

📦 Compilando programas de teste (Gerando LLVM IR, Assembly e Binarios)...
-> Processando factor.pas
AST completa construida com sucesso!
1 warning generated.
1 warning generated.
-> Processando isprime.pas
AST completa construida com sucesso!
1 warning generated.
1 warning generated.
-> Processando pidigits.pas
AST completa construida com sucesso!
1 warning generated.
1 warning generated.
-> Processando fibonacci.pas
AST completa construida com sucesso!
1 warning